# CofC 2026 Staff Match Publication

This staff-only notebook runs the complete reviewed publication workflow directly from Google Drive. GitHub supplies tested code; Drive supplies the match bundle; Supabase stores the evidence, archive, Match Flow, and published COUG scores.

Every production write has its own explicit switch and confirmation text. Leave all write switches `False` until the preceding preview is correct.

## 1. Connect Google Drive

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local Jupyter session detected; Drive mount skipped.')

## 2. Match and reviewer setup

For a new match, normally change only `MATCH_SLUG`, `REVIEWED_BY`, and `APPROVAL_NOTES`.

In [ ]:
SEASON = '2026'
MATCH_SLUG = '2026-08-23_fgcu'
DRIVE_PIPELINE_ROOT = Path('/content/drive/.shortcut-targets-by-id/1CX5Tm9R4U5YA8vJCOOJ4FgMfKIqeQV1D/CofC_Soccer/data_ingestion_pipeline/2026_pipeline')

REVIEWED_BY = 'Anissa Williams'
APPROVAL_NOTES = ''
APPROVE_SOURCE_ARCHIVE = True
APPROVE_MATCH_ANALYTICS = True
APPROVE_COUG_SCORING = True

MATCH_DIR = DRIVE_PIPELINE_ROOT / SEASON / 'matches' / MATCH_SLUG
SOURCE_DIR = MATCH_DIR / '00_source'
STAFF_DIR = MATCH_DIR / 'staff'
BUNDLE_DIR = MATCH_DIR / '20_generated'
print('Match folder:', MATCH_DIR)

## 3. Load current tested code and staff secrets

In Colab, add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY` under the key icon (Secrets). Grant this notebook access to both. Secret values are never printed or written to Drive.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/anissawilliams/cofc-soccer-analytics.git'
if IN_COLAB:
    REPO_ROOT = Path('/content/cofc-soccer-analytics')
    if REPO_ROOT.exists():
        subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', 'main'], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements-backend.txt'), 'rapidfuzz>=3.0.0'], check=True)
    from google.colab import userdata
    missing_secrets = []
    for secret_name in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY'):
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
        else:
            missing_secrets.append(secret_name)
    if missing_secrets:
        raise RuntimeError('Add these Colab Secrets and grant notebook access: ' + ', '.join(missing_secrets))
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((path for path in candidates if (path / 'pipeline').is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError('Open this notebook from the cofc-soccer-analytics repository.')
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')

print('Code revision:', subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip())
print('Supabase credentials: configured')

## 4. Inspect the generated bundle

In [ ]:
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import Markdown, display

REPORT_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_intake_report.json'
VALIDATION_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_validation_report.md'
APPROVAL_PATH = BUNDLE_DIR / f'{MATCH_SLUG}_approval.json'
required_paths = [SOURCE_DIR, BUNDLE_DIR, REPORT_PATH, VALIDATION_PATH, APPROVAL_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Missing required match files:\n' + '\n'.join(missing_paths))
if (SOURCE_DIR / '20_generated').exists():
    print('WARNING: ignore the nested 00_source/20_generated folder; this notebook uses the top-level bundle.')

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
approval = json.loads(APPROVAL_PATH.read_text(encoding='utf-8'))
if report.get('slug') != MATCH_SLUG or str(report.get('season')) != SEASON:
    raise ValueError('Bundle slug/season does not match the setup cell.')
display(Markdown(VALIDATION_PATH.read_text(encoding='utf-8')))
print('Current approval:')
print(json.dumps(approval, indent=2))

## 5. Record staff approval

Review the report above first. Set `WRITE_APPROVAL = True`, run this cell once, then return it to `False`.

In [ ]:
WRITE_APPROVAL = False

if WRITE_APPROVAL:
    if not REVIEWED_BY.strip():
        raise ValueError('REVIEWED_BY is required.')
    report_sha256 = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
    approval_payload = {
        'schema_version': 1,
        'match_slug': MATCH_SLUG,
        'season': SEASON,
        'intake_report_sha256': report_sha256,
        'reviewed_by': REVIEWED_BY.strip(),
        'reviewed_at': datetime.now(ZoneInfo('America/New_York')).isoformat(),
        'approvals': {
            'source_archive': bool(APPROVE_SOURCE_ARCHIVE),
            'match_analytics': bool(APPROVE_MATCH_ANALYTICS),
            'coug_scoring': bool(APPROVE_COUG_SCORING),
        },
        'notes': APPROVAL_NOTES.strip(),
    }
    APPROVAL_PATH.write_text(json.dumps(approval_payload, indent=2, sort_keys=True), encoding='utf-8')
    print('Approval recorded:', APPROVAL_PATH)
    print(json.dumps(approval_payload, indent=2))
else:
    print('Approval unchanged. Set WRITE_APPROVAL = True only after review.')

## 6. Prepare staff events and run all preflight checks

If `staff/staff_events.csv` exists, this roster-matches and validates every incident and writes reviewed staff artifacts into `20_generated`. It then validates the approval, source hashes, archive plan, athlete matching, session/match plan, minutes, and event load. It makes no database changes.

In [ ]:
def run_command(label, command):
    print(f'\n=== {label} ===')
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('PASS' if result.returncode == 0 else f'FAILED (exit {result.returncode})')
    return result

PROMOTE = REPO_ROOT / 'pipeline' / 'ingestion' / 'promote_match_intake.py'
LOAD = REPO_ROOT / 'pipeline' / 'ingestion' / 'load_match.py'
PREPARE_STAFF = REPO_ROOT / 'pipeline' / 'ingestion' / 'prepare_staff_events.py'
LOAD_STAFF = REPO_ROOT / 'pipeline' / 'ingestion' / 'load_staff_events.py'
PUBLISH = REPO_ROOT / 'pipeline' / 'analytics' / 'publish_event_derived_coug_scores.py'

STAFF_CSV = STAFF_DIR / 'staff_events.csv'
STAFF_SUPPLIED = STAFF_CSV.is_file()
if STAFF_SUPPLIED:
    staff_prepare = run_command('Staff event validation', [
        sys.executable, str(PREPARE_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
        '--staff-dir', str(STAFF_DIR), '--output-dir', str(BUNDLE_DIR),
    ])
    staff_prepare_ok = staff_prepare.returncode == 0
else:
    print('No staff/staff_events.csv supplied; incident stage will be skipped.')
    staff_prepare_ok = True

archive_preview = run_command('Archive and artifact preview', [
    sys.executable, str(PROMOTE), '--source-dir', str(SOURCE_DIR), '--bundle-dir', str(BUNDLE_DIR),
])
evidence_preview = run_command('Database evidence preview', [
    sys.executable, str(LOAD), '--slug', MATCH_SLUG, '--season', SEASON,
    '--bundle-dir', str(BUNDLE_DIR), '--dry-run',
])
PREFLIGHT_OK = staff_prepare_ok and archive_preview.returncode == 0 and evidence_preview.returncode == 0
print('\nPreflight status:', 'READY' if PREFLIGHT_OK else 'STOP — resolve failures before applying')

## 7. Load database evidence

This creates/updates the session, match, official stints, and athlete-event evidence. It still does not publish COUG scores.

In [ ]:
APPLY_EVIDENCE = False
EVIDENCE_CONFIRMATION = ''  # type the exact MATCH_SLUG

if not APPLY_EVIDENCE:
    print('Evidence load skipped.')
elif not globals().get('PREFLIGHT_OK', False):
    print('STOP: run a passing preflight first.')
elif EVIDENCE_CONFIRMATION != MATCH_SLUG:
    print(f'STOP: EVIDENCE_CONFIRMATION must equal {MATCH_SLUG!r}.')
else:
    evidence_apply = run_command('APPLY database evidence', [
        sys.executable, str(LOAD), '--slug', MATCH_SLUG, '--season', SEASON,
        '--bundle-dir', str(BUNDLE_DIR),
    ])
    if evidence_apply.returncode != 0:
        print('STOP: evidence load failed; do not continue to archive or scoring.')

## 8. Preview and apply reviewed staff events

Run this after the evidence load. It previews every manual incident before any write. Yellow cards are `-0.4` each; red cards are `-2`. If the preview reports a missing metric, run the matching SQL migration in Supabase before continuing.

In [ ]:
APPLY_STAFF_EVENTS = False
STAFF_CONFIRMATION = ''  # example: APPLY STAFF 2026-08-23_fgcu

STAFF_APPLY_OK = not globals().get('STAFF_SUPPLIED', False)
if not globals().get('STAFF_SUPPLIED', False):
    print('No staff events supplied; nothing to apply.')
else:
    staff_preview = run_command('Staff event database preview', [
        sys.executable, str(LOAD_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
        '--staff-dir', str(STAFF_DIR),
    ])
    STAFF_PREVIEW_OK = staff_preview.returncode == 0
    expected_staff_confirmation = f'APPLY STAFF {MATCH_SLUG}'
    if not STAFF_PREVIEW_OK:
        print('STOP: staff event preview failed; no staff events were applied.')
    elif not APPLY_STAFF_EVENTS:
        print('Staff events previewed only. Review the counts, then use the apply gate.')
    elif STAFF_CONFIRMATION != expected_staff_confirmation:
        print(f'STOP: STAFF_CONFIRMATION must equal {expected_staff_confirmation!r}.')
    else:
        staff_apply = run_command('APPLY staff events', [
            sys.executable, str(LOAD_STAFF), '--season', SEASON, '--slug', MATCH_SLUG,
            '--staff-dir', str(STAFF_DIR), '--apply',
        ])
        STAFF_APPLY_OK = staff_apply.returncode == 0
        if not STAFF_APPLY_OK:
            print('STOP: staff event apply failed; do not archive or publish scores.')

## 9. Archive sources and publish Match Flow

This uploads the approved source and generated artifacts to Supabase Storage and registers them to the match session. The backend reads the promoted Match Flow from there—no Git copy is needed.

In [ ]:
APPLY_ARCHIVE = False
ARCHIVE_CONFIRMATION = ''  # type the exact MATCH_SLUG

if not APPLY_ARCHIVE:
    print('Archive promotion skipped.')
elif not globals().get('PREFLIGHT_OK', False):
    print('STOP: run a passing preflight first.')
elif globals().get('STAFF_SUPPLIED', False) and not globals().get('STAFF_APPLY_OK', False):
    print('STOP: supplied staff events must be successfully applied first.')
elif ARCHIVE_CONFIRMATION != MATCH_SLUG:
    print(f'STOP: ARCHIVE_CONFIRMATION must equal {MATCH_SLUG!r}.')
else:
    archive_apply = run_command('APPLY archive and Match Flow', [
        sys.executable, str(PROMOTE), '--source-dir', str(SOURCE_DIR),
        '--bundle-dir', str(BUNDLE_DIR), '--apply',
    ])

## 10. Preview COUG scores

Run this after the evidence load. It prints every proposed player score and writes the review CSV into Drive. It does not update the public table.

In [ ]:
SCORE_REVIEW_DIR = BUNDLE_DIR / 'score_review'
if globals().get('STAFF_SUPPLIED', False) and not globals().get('STAFF_APPLY_OK', False):
    print('STOP: apply the supplied staff events before previewing final scores.')
    SCORE_PREVIEW_OK = False
else:
    score_preview = run_command('COUG score preview', [
        sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
        '--output-root', str(SCORE_REVIEW_DIR),
    ])
    SCORE_PREVIEW_OK = score_preview.returncode == 0
print('Score preview:', 'READY FOR STAFF DECISION' if SCORE_PREVIEW_OK else 'STOP — scores are not publishable')

## 11. Publish the reviewed COUG scores

Compare the preview with the match and video first. To publish, set the switch and type `PUBLISH <match slug>` exactly.

In [ ]:
PUBLISH_COUG_SCORES = False
SCORE_CONFIRMATION = ''  # example: PUBLISH 2026-08-23_fgcu

expected_confirmation = f'PUBLISH {MATCH_SLUG}'
if not PUBLISH_COUG_SCORES:
    print('COUG score publication skipped.')
elif not globals().get('SCORE_PREVIEW_OK', False):
    print('STOP: run a successful score preview first.')
elif SCORE_CONFIRMATION != expected_confirmation:
    print(f'STOP: SCORE_CONFIRMATION must equal {expected_confirmation!r}.')
else:
    fresh_preview = run_command('Fresh COUG score preview', [
        sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
        '--output-root', str(SCORE_REVIEW_DIR),
    ])
    if fresh_preview.returncode != 0:
        print('STOP: fresh score preview failed; nothing published.')
    else:
        score_apply = run_command('PUBLISH COUG scores', [
            sys.executable, str(PUBLISH), '--season', SEASON, '--slug', MATCH_SLUG,
            '--output-root', str(SCORE_REVIEW_DIR), '--apply',
        ])

## 12. Verify Supabase publication

In [ ]:
from supabase import create_client
import pandas as pd

client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])
match_date = MATCH_SLUG[:10]
sessions = client.table('session').select('id,session_date,season,competition,notes').eq('session_date', match_date).eq('season', SEASON).execute().data or []
sessions = [row for row in sessions if f'slug: {MATCH_SLUG}' in str(row.get('notes') or '').splitlines()]
if len(sessions) != 1:
    print(f'STOP: expected one session for {MATCH_SLUG}; found {len(sessions)}')
else:
    session_id = sessions[0]['id']
    matches = client.table('match').select('result,goals_for,goals_against').eq('session_id', session_id).execute().data or []
    scores = client.table('coug_score').select('athlete:athlete_id(display_name),aset_score,peak_score,set_piece_score,positional_score,load_score,total_score,score_type,weight:weight_version_id(version)').eq('session_id', session_id).eq('score_type', 'match').execute().data or []
    scores = [row for row in scores if (row.get('weight') or {}).get('version') == 'trial_1']
    artifacts = client.table('source_file').select('source_type,original_filename,upload_status,parse_status').eq('session_id', session_id).eq('is_active', True).execute().data or []
    print('Session:', sessions[0])
    print('Match:', matches)
    print('Published score rows:', len(scores))
    if scores:
        display(pd.DataFrame([{**row, 'player': (row.get('athlete') or {}).get('display_name')} for row in scores]).drop(columns=['athlete', 'weight'], errors='ignore').sort_values('total_score', ascending=False))
    print('Registered artifacts:', len(artifacts))
    display(pd.DataFrame(artifacts))

## Finished

Keep the approval, promotion receipt, and score-review CSV in the match's `20_generated` folder. If any step fails, stop at that stage; rerunning successful apply steps is designed to be idempotent.